In [0]:
-- vendor scorecard

CREATE OR REPLACE MATERIALIZED VIEW gold_vendor_scorecard
AS
SELECT
    v.vendor_id,
    v.vendor_name,
    COUNT(DISTINCT p.category_name)  AS categories_sold,
    COUNT(DISTINCT f.item_id)        AS products_sold,
    COUNT(DISTINCT f.store_id)       AS stores_served,
    SUM(f.sale_dollars)              AS total_revenue,
    SUM(f.bottles_sold)              AS total_bottles,
    SUM(f.gross_profit)              AS total_profit,
    RANK() OVER (ORDER BY SUM(f.sale_dollars) DESC) AS vendor_rank
FROM dev.silver.fct_sales_deduped f
JOIN dev.silver.dim_vendors v
    ON  f.vendor_id = v.vendor_id
    AND f.sale_date >= v.__START_AT
    AND (f.sale_date < v.__END_AT OR v.__END_AT IS NULL)
JOIN dev.silver.dim_products p
    ON  f.item_id = p.item_id
    AND f.sale_date >= p.__START_AT
    AND (f.sale_date < p.__END_AT OR p.__END_AT IS NULL)
GROUP BY v.vendor_id, v.vendor_name

In [0]:
-- category market share

CREATE OR REPLACE MATERIALIZED VIEW gold_category_market_share
AS
WITH category_totals AS (
    SELECT
        p.category_name,
        SUM(f.sale_dollars)    AS category_revenue,
        SUM(f.bottles_sold)    AS category_bottles,
        COUNT(*)               AS category_transactions
    FROM dev.silver.fct_sales_deduped f
    JOIN dev.silver.dim_products p
        ON  f.item_id = p.item_id
        AND f.sale_date >= p.__START_AT
        AND (f.sale_date < p.__END_AT OR p.__END_AT IS NULL)
    GROUP BY p.category_name
),
grand_total AS (
    SELECT SUM(category_revenue) AS total_revenue
    FROM category_totals
)
SELECT
    ct.category_name,
    ct.category_revenue,
    ct.category_bottles,
    ct.category_transactions,
    ROUND(ct.category_revenue / gt.total_revenue * 100, 2) AS market_share_pct,
    RANK() OVER (ORDER BY ct.category_revenue DESC) AS category_rank
FROM category_totals ct
CROSS JOIN grand_total gt